In [1]:

from pathlib import Path
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix


BASE_DIR = Path.cwd()

if BASE_DIR.name.lower() == "data":
    DATA_DIR = BASE_DIR
else:
    DATA_DIR = BASE_DIR / "Data"

CSV_PATH = DATA_DIR / "data.csv"

if not CSV_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the dataset at: {CSV_PATH}\n"
        f"Current working directory: {BASE_DIR}\n"
        "Check that data.csv exists inside your Data folder."
    )

df = pd.read_csv(CSV_PATH)






df = df.drop(columns=["id"], errors="ignore")
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})

if df["diagnosis"].isna().any():
    raise ValueError(
        "Some diagnosis values could not be mapped. "
        "Check that the diagnosis column contains M and B."
    )





X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]


# ============================================================
# 3. SPLIT THE DATA
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


# ============================================================
# 4. SELECT THE TOP 5 FEATURES
# ============================================================

train_for_corr = X_train.copy()
train_for_corr["diagnosis"] = y_train

correlations = (
    train_for_corr.corr(numeric_only=True)["diagnosis"]
    .drop("diagnosis")
    .abs()
    .sort_values(ascending=False)
)

selected_features = correlations.head(5).index.tolist()

print("Selected features:")
for feature in selected_features:
    print("-", feature)


X_train_5 = X_train[selected_features]
X_test_5 = X_test[selected_features]



# ============================================================
# 5. TRAIN THE MODEL
# ============================================================

final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

final_model.fit(X_train_5, y_train)


# ============================================================
# 7. PREPARE DATA FOR VISUALIZATIONS
# ============================================================

selected_features_df = X_train_5.copy()

selected_features_df["diagnosis"] = y_train.map({
    0: "Benign",
    1: "Malignant"
})


# ============================================================
# 8. SAVE THE MODEL AND REQUIRED DATA
# ============================================================

DATA_DIR.mkdir(parents=True, exist_ok=True)

with open(DATA_DIR / "final_model.pkl", "wb") as f:
    pickle.dump(final_model, f)

with open(DATA_DIR / "selected_features.pkl", "wb") as f:
    pickle.dump(selected_features, f)

with open(DATA_DIR / "selected_features_df.pkl", "wb") as f:
    pickle.dump(selected_features_df, f)




Selected features:
- concave points_worst
- perimeter_worst
- radius_worst
- concave points_mean
- perimeter_mean
